# Derivations at any granularity

Grow a small world and choose a call to an operator in its second layer.

In [1]:
import random
from opulse.config import Settings
from opulse.core.starting_sets import starting_registry
from opulse.definition.parser import DefinitionParser
from opulse.expression.nodes import Call, Number
from opulse.expression.render import ExpressionRenderer
from opulse.generation.pipeline import OperatorFactory, build_layers
from opulse.trace import Reducer
from opulse.verify import Oracle, verify_trace

registry = starting_registry()
settings = Settings()
factory = OperatorFactory(settings, registry, random.Random(13))
build_layers(factory, [4, 2])
chosen = next(op for op in registry if op.definition and op.recursive_layer == 1)
expression = Call(chosen, tuple(Number(3) for _ in range(chosen.n_ary)))
parser = DefinitionParser(settings.symbols, registry)
renderer = ExpressionRenderer(settings.symbols)
print(renderer.full(expression))

3 ●₁₉


Compare full unfolding with two sets of known operators and check every trace.

In [2]:
oracle = Oracle(registry)
known_sets = {
    "none": set(),
    "first layer": {op.symbol for op in registry if op.definition and op.recursive_layer == 0},
    "chosen operator": {chosen.symbol},
}
traces = {}
for label, known in known_sets.items():
    reducer = Reducer(registry, factory.runtime, parser,
                      expand=lambda op: op.definition is not None and op.symbol not in known)
    trace = reducer.reduce(expression)
    assert not trace.truncated
    verify_trace(trace, oracle)
    traces[label] = trace
    print(label, "steps:", len(trace.steps),
          "expansions:", sum(step.kind == "expand" for step in trace.steps),
          "value:", trace.value)
assert len({trace.value for trace in traces.values()}) == 1

none steps: 13 expansions: 4 value: 0
first layer steps: 13 expansions: 4 value: 0
chosen operator steps: 1 expansions: 0 value: 0


Show the first few checked rewrites in the full trace.

In [3]:
for step in traces["none"].steps[:5]:
    print(step.kind, renderer.full(step.expression_after))

expand (((3 % 5) - 1) ●₁₉) $
eval ((3 - 1) ●₁₉) $
eval (2 ●₁₉) $
expand ((((2 % 5) - 1) ●₁₉) $) $
eval (((2 - 1) ●₁₉) $) $
